# CREDIT AND DEBIT CARD FRAUD DETECTION

# INTRODUCTION

In 2024 alone, 62 million Americans were victims of fraudulent charges on their credit or debit cards, with unauthorized purchases exceeding $6.2 billion annually (Cruz, 2025). Additionally, nearly one in three Americans report being a victim of online financial fraud or cybercrime (Newall & Deeney, 2023). With credit and debit cards now accounting for 60% of all payments (a figure that continues to rise), and with the shift toward a cashless society, opportunities for fraud will only increase (Cruz, 2025). As such, detecting fraud early and practicing card safety are more important than ever. 

Hence, the primary purpose of this project is to develop a machine learning model to accurately classify credit and debit card transactions as fraudulent or legitimate. In order to do so, this project uses a comprehensive financial transactions dataset which contains transaction records, customer information, card data, and fraudulence labels from a banking institution, spanning across the 2010s decade (Kaggle, 2024). 

Beyond building the model, this project also aims to investigate the influence of several factors on fraud risk by addressing the following questions:

1. Which state is most impacted by fraud

2. How are debit/credit card fraud rates related to income level?

3. Are men or women most impacted by fraud?

# PROJECT WORKFLOW

Below are the general steps followed in this project workflow:

1. Gathered data from Kaggle
   
    - An overview of this data is provided below in the "raw data overview" section

2. Cleaned/transformed data using Pandas

   - Details/code necessary for this step are provided in the "notebooks/01_etl.ipynb" file

3. Created SQL schema (DDL script) to load cleaned data into

   - Details/code necessary for this step are provided in the "sql/schema.sql", and "sql/ERD_diagram.png" files

4. Loaded cleaned data into SQL database

   - Details/code necessary for step are provided in the "notebooks/01_etl.ipynb", "sql/materialized_view.sql", "sql/schema.sql", and "sql/view.sql" files

5. Joined SQL tables together and saved all joined data to csv

   - Details/code necessary for this step are provided in the "notebooks/01_etl.ipynb" file

6. Developed ML models (along with EDA for the models) using csv created in previous step

   - Details/code necessary for this step are provided in the "notebooks/02_model.ipynb" and "fraud_detection_pipeline.joblib" files

7. Created dashboard using data from the SQL database

   - Details/code necessary for this step are provided in the "dashboard.py"



# ELT PROCESS SUMMARY

Data Transformation
1. Transactions data cleaning

   a. Handled the large transaction file in chunks

   b. Standardized column names, convert date strings to datetime objects, removed 
      dollar signs, standardized state codes, etc.

2. Fraud labels cleaning

   a. Parsing JSON structure

   b. Converted text "Yes"/"No" labels to binary 1/0 values

3. Cards data cleaning

   a. Standardized card brands and types

   b. Created binary flags for features like "has_chip" and "is_prepaid"

   c. Extracted expiration and account opening dates into separate year/month 
      columns

   d. Removed unnecessary fields, including sensitive data

4. Users data cleaning

   a. Removed currency symbols and commas from monetary fields

   b. Median imputation for missing values

   c. Made sure credit scores were in the range 300-850

Data Loading

1. Created a schema

   a. Defined primary keys

   b. Set up foreign key relationships between tables

   c. Created indexes on frequently queried columns

2. Wrote a Python function to load data in chunks, and used transactions

3. Created a view (fraud_analysis_view) that joined the transactions (including labels 
   from the JSON file that were joined in Python), users, and cards tables

   a. Added derived time fields (hour, day of week, month, year)

4. Created a materialized view (fraud_analysis_table) that persisted the view data as a 
   physical table for better query performance, and added additional indexes for faster 
   querying

Chunked Processing

1. Breaking the large transaction file into chunks allowed me to process a file that might 
  otherwise exceed available memory

2. Used the `estimate_chunk_size` function when processing the transactions to size the 
  chunks based on row memory usage

Materialized Views

1. The materialized view 'fraud_analysis_table' offers an advantage of query performance 
   by pre-joining the tables and storing the result as a physical table.

2. When doing analysis, we can query a single table with all relevant fields rather than 
   writing repeated join statements.

3. The join is only called once during the materialization process rather than repeatedly 
   during analysis.

Indexing Strategy

1. Transaction-focused indexes on `date`, `amount`, and `is_fraud` flag

2. Relationship indexes on `client_id` and `card_id`

3. Composite indexes `(date, is_fraud)` to support potential query patterns

4. Additional indexes on the materialized view to optimize the analytical queries

This indexing strategy is important in a context where analysts need to quickly search for 
patterns in a large-volume transaction table. It also accounts for common queries, like 
time-based queries and filtering by fraud status.

# RAW DATA OVERVIEW

As mentioned, the data used in this project was sourced from Kaggle. It contains 5 files - transactions_data.csv, cards_data.csv, users_data.csv, mcc_codes.json, and train_fraud_labels.json. Below, each will be introduced:

TRANSACTIONS DATA

The transactions dataset is the primary dataset of this project. It is a csv file that contains various information about card transactions including the type of transaction and its location and contains 13305915 rows and 12 columns:

In [2]:
import pandas as pd

transactions_data = pd.read_csv("../data/transactions_data.csv")

print(transactions_data.head())
print(transactions_data.describe())
print(transactions_data.info())

        id                 date  client_id  card_id   amount  \
0  7475327  2010-01-01 00:01:00       1556     2972  $-77.00   
1  7475328  2010-01-01 00:02:00        561     4575   $14.57   
2  7475329  2010-01-01 00:02:00       1129      102   $80.00   
3  7475331  2010-01-01 00:05:00        430     2860  $200.00   
4  7475332  2010-01-01 00:06:00        848     3915   $46.41   

            use_chip  merchant_id merchant_city merchant_state      zip   mcc  \
0  Swipe Transaction        59935        Beulah             ND  58523.0  5499   
1  Swipe Transaction        67570    Bettendorf             IA  52722.0  5311   
2  Swipe Transaction        27092         Vista             CA  92084.0  4829   
3  Swipe Transaction        27092   Crown Point             IN  46307.0  4829   
4  Swipe Transaction        13051       Harwood             MD  20776.0  5813   

  errors  
0    NaN  
1    NaN  
2    NaN  
3    NaN  
4    NaN  
                 id     client_id       card_id   merchant_id 

Column descriptions:

id: A unique identifier for each transaction

date: The date/time each transaction took place

client_id: The id number for the card holder

card_id: The id number for the card used

amount: The amount used in the transaction

use_chip: The type of transaction (swipe, chip, or online)

merchant_id: The id number for the merchant where the transaction took place

merchant_city: The city of the merchant where the transaction took place

merchant_state: The state of the merchant where the transaction took place

zip: The zip code of the merchant where the transaction took place

mcc: Merchant category codes (four-digit numbers used to classify businesses based on their industry or type of service)

errors: Tracks if there were any errors with the processing of the transaction (column description was not well documented)

CARDS DATA

The cards dataset is one of the supplementary datasets in this project. It is a csv file that contains information about the credit/debit cards used in the transactions and has 6146 rows and 13 columns:

In [3]:
cards_data = pd.read_csv("../data/cards_data.csv")

print(cards_data.head())
print(cards_data.describe())
print(cards_data.info())

     id  client_id  card_brand        card_type       card_number  expires  \
0  4524        825        Visa            Debit  4344676511950444  12/2022   
1  2731        825        Visa            Debit  4956965974959986  12/2020   
2  3701        825        Visa            Debit  4582313478255491  02/2024   
3    42        825        Visa           Credit  4879494103069057  08/2024   
4  4659        825  Mastercard  Debit (Prepaid)  5722874738736011  03/2009   

   cvv has_chip  num_cards_issued credit_limit acct_open_date  \
0  623      YES                 2       $24295        09/2002   
1  393      YES                 2       $21968        04/2014   
2  719      YES                 2       $46414        07/2003   
3  693       NO                 1       $12400        01/2003   
4   75      YES                 1          $28        09/2008   

   year_pin_last_changed card_on_dark_web  
0                   2008               No  
1                   2014               No  
2       

Column descriptions:

id: The unique id number of the card

client_id: The id number of the card holder

card_brand: The brand of the card

card_type: The type of card 

card_number: The number of the card

expires: The card expiration date

cvv: The card cvv number

has_chip: Whether the card has a chip or not

num_cards_issued: While the description of this row is not completely clear, it can be assumed it refers to how many physical cards have been issued for that specific card account

credit_limit: The credit limit of the card

acct_open_date: The date the card account was opened

year_pin_last_changed: The year the card pin was last changed

card_on_dark_web: Whether or not the card is on the dark web

USERS DATA

The users data is the second supplementary dataset used in this project. It is a csv file that contains information about the card holders of the transactions and has 2000 rows and 14 columns:

In [4]:
users_data = pd.read_csv("../data/users_data.csv")

print(users_data.head())
print(users_data.describe())
print(users_data.info())

     id  current_age  retirement_age  birth_year  birth_month  gender  \
0   825           53              66        1966           11  Female   
1  1746           53              68        1966           12  Female   
2  1718           81              67        1938           11  Female   
3   708           63              63        1957            1  Female   
4  1164           43              70        1976            9    Male   

                    address  latitude  longitude per_capita_income  \
0             462 Rose Lane     34.15    -117.76            $29278   
1    3606 Federal Boulevard     40.76     -73.74            $37891   
2           766 Third Drive     34.02    -117.89            $22681   
3          3 Madison Street     40.71     -73.99           $163145   
4  9620 Valley Stream Drive     37.76    -122.44            $53797   

  yearly_income total_debt  credit_score  num_credit_cards  
0        $59696    $127613           787                 5  
1        $77254   

Column descriptions:

id: A unique id for each user

current_age: The current age of the card holder

retirement_age: The retirement age of the card holder

birth_year: The birth year of the card holder

birth_month: The birth month of the card holder

gender: The gender of the card holder

address: The address of the card holder

latitude: The latitude of the card holder's address

longitude: The longitude of the card holder's address

per_capita_income: This column description was not well documented. Can assume it refers to the per capita income of the people in that region.

yearly_income: The yearly income of the card holder

total_debt: The total debt the card holder has

credit_score: The credit score of the card holder

num_credit_cards: The number of credit cards the card holder has

MCC CODES DATA

The mcc codes data is a json file that contains four-digit numbers used to classify businesses based on their industry or type of service known as "merchant category codes" and has 109 rows and 2 columns:

Note: Although the information in this table wasn't actually used in this project, the data is still displayed here to view

In [7]:
import json

with open("../data/mcc_codes.json", "r") as f:
    mcc_json = json.load(f)
    
mcc_codes = pd.DataFrame(list(mcc_json.items()), columns=["mcc_code", "mcc_description"])

print(mcc_codes.head())
print(mcc_codes.describe())
print(mcc_codes.info())

  mcc_code                       mcc_description
0     5812         Eating Places and Restaurants
1     5541                      Service Stations
2     7996  Amusement Parks, Carnivals, Circuses
3     5411          Grocery Stores, Supermarkets
4     4784                 Tolls and Bridge Fees
       mcc_code     mcc_description
count       109                 109
unique      109                 108
top        5812  Passenger Railways
freq          1                   2
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   mcc_code         109 non-null    object
 1   mcc_description  109 non-null    object
dtypes: object(2)
memory usage: 1.8+ KB
None


Column descriptions:

mcc_code: A four-digit code used to classify businesses based on their industry or type of service 

mcc_description: The description of the code

FRAUD LABELS DATA

The fraud labels data is a json file that contains labels (either yes or no) that state whether a transaction was fraudulent or not and has 8914963 rows and 2 columns:

In [9]:
with open("../data/train_fraud_labels.json", "r") as f:
    labels_json = json.load(f)

fraud_labels = labels_json["target"]

fraud_labels = pd.DataFrame(list(fraud_labels.items()), columns=["fraud_label_id", "fraud_labels"])

print(fraud_labels.head())
print(fraud_labels.describe())
print(fraud_labels.info())

  fraud_label_id fraud_labels
0       10649266           No
1       23410063           No
2        9316588           No
3       12478022           No
4        9558530           No
       fraud_label_id fraud_labels
count         8914963      8914963
unique        8914963            2
top          15151926           No
freq                1      8901631
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8914963 entries, 0 to 8914962
Data columns (total 2 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   fraud_label_id  object
 1   fraud_labels    object
dtypes: object(2)
memory usage: 136.0+ MB
None


Column descriptions:

fraud_label_id: A unique identifier for each fraud label

fraud_labels: Whether the transaction was fraudulent or not

# CONCLUSIONS

The following are the main conclusions that were reach from both the machine learning and the questions of interest:

ML Results:

Best Performing Model:

Logistic Regression + SMOTE

Results:
Recall (Fraud): 60% — 149 out of 248 frauds correctly identified


Precision (Fraud): ~0.03% — Very high false positive rate (~463k false alarms)


PR AUC: 0.0005 — Very low due to poor fraud precision


Interpretation:

SMOTE improved recall — the model is better at catching fraud compared to LR w/out SMOTE


But precision dropped dramatically — flagged too many non-fraud cases as fraud


The model is not ready for production but demonstrates how class imbalance impacts model performance


This is a strong proof of concept for detecting rare events like fraud

Questions Results:

1. Which state is most impacted by fraud?

Seems to be Ohio

2. How are debit/credit card fraud rates related to income level?

Credit/debit card fraud tends to target lower-income clients, but varies across levels of debt.

3. Are men or women most impacted by fraud?

Looks to be women

# REFERENCES

ComputingVictor. (2024). 💳 Financial Transactions Dataset: Analytics. Kaggle.com. https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets/data?select=train_fraud_labels.json

Cruz, B. (2025, January 27). Credit Card Fraud 2021 Annual Report: Prevalence, Awareness, and Prevention. Security.org. https://www.security.org/digital-safety/credit-card-fraud-report/

Newall, M., & Deeney, C. (2023, December). Nearly 1 in 3 Americans report being a victim of online financial fraud or cybercrime. Ipsos. https://www.ipsos.com/en-us/nearly-1-3-americans-report-being-victim-online-financial-fraud-or-cybercrime